# Databricks Lakehouse - Gold Layer Business Aggregations (`03_create_gold`)

## Overview
This notebook reads cleaned telemetry data from the **Silver Layer** (`workspace.neuro_insight.silver_telemetry`) and creates daily business-level aggregations stored in the **Gold Layer** (`workspace.neuro_insight.gold_daily_metrics`).

### Core Transformations:
- **Daily Patient Aggregations**: Computes daily averages for tremor scores, reaction times, gait speeds, cognitive scores, and heart rates.
- **Data Completeness Tagging**: Categorizes days as `COMPLETE` or `INCOMPLETE` based on total measurement counts per day.
- **Clinical Severity Categorization**: Assigns a `tremor_band` (`LOW`, `MODERATE`, `HIGH`) derived from the aggregated daily tremor score.

---

### Step 1: Environment Setup & Silver Data Loading
Import PySpark functions and load validated telemetry records from the Silver Delta table.

In [ ]:
from pyspark.sql import functions as F

# Read cleaned data from Silver layer
silver_df = spark.table("workspace.neuro_insight.silver_telemetry")

print("Total Silver records loaded:", silver_df.count())

### Step 2: Extract Event Date & Aggregate Daily Metrics
Group records by `patient_id` and `event_date` to calculate total measurement counts and rounded daily averages for key clinical indicators.

In [ ]:
# Extract event date from event timestamp
gold_source_df = silver_df.withColumn("event_date", F.to_date("event_timestamp"))

# Aggregate daily patient metrics
gold_daily_df = (
    gold_source_df
    .groupBy("patient_id", "event_date")
    .agg(
        F.count("*").alias("measurement_count"),
        F.round(F.avg("tremor_score"), 2).alias("avg_tremor_score"),
        F.round(F.avg("reaction_time_ms"), 2).alias("avg_reaction_time_ms"),
        F.round(F.avg("gait_speed_mps"), 2).alias("avg_gait_speed_mps"),
        F.round(F.avg("cognitive_score"), 2).alias("avg_cognitive_score"),
        F.round(F.avg("heart_rate_bpm"), 2).alias("avg_heart_rate_bpm")
    )
)

### Step 3: Apply Business Logic & Feature Engineering
Enrich the daily metrics dataset with business classification logic:
1. **`data_completeness`**: Marked as `COMPLETE` if a patient has at least 80 measurements recorded in a day, otherwise `INCOMPLETE`.
2. **`tremor_band`**: Segmented into `LOW` (< 3.0), `MODERATE` (3.0 to 6.99), and `HIGH` (>= 7.0).

In [ ]:
# Add completeness and tremor classification flags
gold_daily_df = (
    gold_daily_df
    .withColumn(
        "data_completeness",
        F.when(F.col("measurement_count") >= 80, "COMPLETE").otherwise("INCOMPLETE")
    )
    .withColumn(
        "tremor_band",
        F.when(F.col("avg_tremor_score") < 3, "LOW")
        .when(F.col("avg_tremor_score") < 7, "MODERATE")
        .otherwise("HIGH")
    )
)

### Step 4: Write Aggregated Features to Gold Delta Table
Save the final enriched Gold table `workspace.neuro_insight.gold_daily_metrics` in Delta format for downstream analytics and BI dashboarding.

In [ ]:
# Save aggregated dataset to Gold Delta table
(
    gold_daily_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.neuro_insight.gold_daily_metrics")
)

print("Gold table workspace.neuro_insight.gold_daily_metrics created successfully.")